# Phase 3 — 04c: SFT arm, short-trace variant (the A/B twin of 04)

**Runtime: L4.** ~1 h, ~5 units. Run AFTER 04b (needs `phase3/traces_v0.json`).

**Everything is identical to notebook 04** — same seed (3407), same mixture
recipe (sft-pile x3, others x1, same 150 restraint functions), same LoRA
(r=32, α=64), same trainer, same dev eval — **except the targets**: where a
verified teacher trace exists (522/544 bugs), the answer is the DeepSeek
diagnosis + its execution-verified fix. Uncovered bugs fall back to plain
gold-fix targets, exactly like 04. That is the whole experimental difference,
so any dev-set difference is attributable to the traces (A2's 1-seed A/B).

**Bonus noise ruler:** we re-measure the untrained base first (`base2`). The
base-vs-base2 difference is pure sampling noise — if the two arms differ by
less than that, the honest verdict is a tie.

In [ ]:
# Setup: Drive, fresh repo, data, routing, teacher traces
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
traces = {t['id']: t for t in json.load(open(f'{PHASE3}/traces_v0.json'))}
train_bugs = [b for b in bugs if b['split'] == 'train']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
print(len(train_bugs), 'train |', len(dev_bugs), 'dev |', len(traces), 'verified traces')

In [ ]:
# Trace-arm mixture: SAME recipe as 04, only the TARGETS differ.
# rng stream matches 04 exactly (same restraint picks, same final shuffle);
# diagnosis templates draw from a separate rng so parity is preserved.
SEED = 3407
rng = random.Random(SEED)
rng2 = random.Random(SEED + 1)

def bug_example(b):
    t = traces.get(b['id'])
    if t:
        answer = t['diagnosis'] + '\n\n```python\n' + t['fix'].strip() + '\n```'
    else:
        answer = '```python\n' + b['fixed'].strip() + '\n```'
    return {'prompt': build_repair_prompt(b['buggy'], b['test_list']),
            'answer': answer}

RESTRAINT_DIAGNOSES = [
    'The function is already correct: it passes every test as written, so no change is needed.',
    'There is no bug here — the logic already satisfies all the tests, so the function stays unchanged.',
    'The code is correct as given; every test passes, so the right fix is no fix at all.',
]

mixture, traced = [], 0
for b in train_bugs:
    copies = 3 if routing.get(b['id']) == 'sft' else 1
    mixture += [bug_example(b)] * copies
    traced += (b['id'] in traces) * copies
clean_train = [r for r in restraint if r['split'] == 'train']
rng.shuffle(clean_train)
for r in clean_train[:150]:
    diag = rng2.choice(RESTRAINT_DIAGNOSES)
    mixture.append({'prompt': build_repair_prompt(r['code'], r['test_list']),
                    'answer': diag + '\n\n```python\n' + r['code'].strip() + '\n```'})
rng.shuffle(mixture)
print(f'{len(mixture)} examples | {traced} bug slots carry a teacher trace | 150 restraint w/ template diagnosis')

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Load model 4-bit + LoRA — identical config to 04
from unsloth import FastLanguageModel
import torch
model, tok = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1024,
    load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)
print('LoRA attached (untrained LoRA == base model)')

In [ ]:
# Dataset in chat format (eyeball one: diagnosis first, then the fenced fix)
from datasets import Dataset

def to_text(ex):
    msgs = [{'role': 'user', 'content': ex['prompt']},
            {'role': 'assistant', 'content': ex['answer']}]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}

ds = Dataset.from_list(mixture).map(to_text)
print('ONE FORMATTED EXAMPLE (eyeball it — diagnosis THEN code in the answer):')
print(ds[0]['text'][:1500])

In [ ]:
# Dev-slice evaluation — identical to 04; extract_code ignores the diagnosis
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

def passes(code, bug, timeout=6):
    prog = '\n'.join(list(bug['test_imports']) + [code] + list(bug['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

@torch.no_grad()
def dev_eval(tag, k=16, batch_prompts=2):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for i in range(0, len(dev_bugs), batch_prompts):
        chunk = dev_bugs[i:i+batch_prompts]
        texts = [tok.apply_chat_template([{'role':'user','content':build_repair_prompt(b['buggy'], b['test_list'])}],
                 tokenize=False, add_generation_prompt=True) for b in chunk]
        enc = tok(texts, return_tensors='pt', padding=True, padding_side='left').to('cuda')
        out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                             num_return_sequences=k, max_new_tokens=512,
                             pad_token_id=tok.eos_token_id)
        gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        for j, b in enumerate(chunk):
            codes = [extract_code(t) for t in gen[j*k:(j+1)*k]]
            with ThreadPoolExecutor(max_workers=4) as pool:
                flags = list(pool.map(lambda c: passes(c, b), codes))
            per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, 'pass@16': p16, 'gap': p16 - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE3}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@16={p16*100:.1f}%  headroom gap={(p16-p1)*100:.1f}")
    return res

base2_res = dev_eval('base2_seed%d' % SEED)   # the noise ruler vs 04's base row

In [ ]:
# TRAIN epoch 1 -> eval -> TRAIN epoch 2 -> eval  (identical trainer to 04)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def make_trainer():
    FastLanguageModel.for_training(model)
    t = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
        dataset_text_field='text',
        args=SFTConfig(per_device_train_batch_size=8, gradient_accumulation_steps=2,
            num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_ratio=0.05, logging_steps=10, seed=SEED, output_dir='/content/out',
            report_to='none', save_strategy='no'))
    return train_on_responses_only(t,
        instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

make_trainer().train()
model.save_pretrained(f'{PHASE3}/sft_trace_s{SEED}_ep1')
ep1_res = dev_eval('trace_ep1_seed%d' % SEED)

make_trainer().train()
model.save_pretrained(f'{PHASE3}/sft_trace_s{SEED}_ep2')
ep2_res = dev_eval('trace_ep2_seed%d' % SEED)

In [ ]:
# THE A/B TABLE: both arms side by side (04's rows loaded from Drive)
rows = []
for tag in ('base_seed3407', 'ep1_seed3407', 'ep2_seed3407'):
    rows.append(('no-trace', json.load(open(f'{PHASE3}/dev_eval_{tag}.json'))))
for res in (base2_res, ep1_res, ep2_res):
    rows.append(('trace', res))
print(f"{'arm':<9} {'tag':<20} pass@1   pass@16   gap")
for arm, r in rows:
    print(f"{arm:<9} {r['tag']:<20} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
b1 = json.load(open(f'{PHASE3}/dev_eval_base_seed3407.json'))
noise = abs(b1['pass@1'] - base2_res['pass@1']) * 100
print(f"\nNOISE RULER: |base - base2| pass@1 = {noise:.1f} pts — differences smaller than this are a tie.")
print('Decision: compare arms at their BEST epoch. Winner = higher pass@16;')
print('tie-break by pass@1; within-noise = tie -> prefer the simpler no-trace arm.')
print('\nAdapters saved to Drive phase3/. Bring the whole table back to the session.')

## Bring back to the session
1. The **full six-row A/B table** from the last cell
2. The **noise-ruler line**
3. The **formatted-example printout** (did the diagnosis+fix targets look right?)
4. Anything unhealthy along the way

After this: the A/B verdict picks the SFT recipe → 3 seeds of the winner
(Phase 3 proper) → forgetting probe → ONE held-out milestone eval.